In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8')
%matplotlib inline

print("✅ Libraries loaded!")

✅ Libraries loaded!


In [2]:
# Load cleaned reviews
df = pd.read_csv('../data/raw/all_reviews.csv')
print(f"✅ Data loaded! Shape: {df.shape}")
print(f"\nReviews per bank:")
print(df['bank'].value_counts())
print(f"\nSample:")
df.head(3)

✅ Data loaded! Shape: (1449, 5)

Reviews per bank:
bank
Bank of Abyssinia              498
Dashen Bank                    495
Commercial Bank of Ethiopia    456
Name: count, dtype: int64

Sample:


,review,rating,date,bank,source
0,🤙🏼🤙🏼,5,2026-05-16,Commercial Bank of Ethiopia,Google Play
1,worst,1,2026-05-16,Commercial Bank of Ethiopia,Google Play
2,this app very full,5,2026-05-15,Commercial Bank of Ethiopia,Google Play


In [3]:
# ============================================
# SENTIMENT ANALYSIS USING VADER
# Tool Selection Rationale:
# - VADER is optimized for short social media text
# - No training required, works out of the box
# - Returns compound score (-1 to +1)
# - Fast enough for 1,449 reviews
# ============================================

analyzer = SentimentIntensityAnalyzer()

def get_sentiment_score(text):
    try:
        return analyzer.polarity_scores(str(text))['compound']
    except:
        return 0.0

def classify_sentiment(score):
    if score > 0.05:
        return 'positive'
    elif score < -0.05:
        return 'negative'
    else:
        return 'neutral'

print("Running sentiment analysis...")
df['sentiment_score'] = df['review'].apply(get_sentiment_score)
df['sentiment_label'] = df['sentiment_score'].apply(classify_sentiment)

print("✅ Sentiment analysis complete!")
print(f"\nOverall sentiment distribution:")
print(df['sentiment_label'].value_counts())
print(f"\nSentiment score statistics:")
print(df['sentiment_score'].describe().round(4))

Running sentiment analysis...
✅ Sentiment analysis complete!

Overall sentiment distribution:
sentiment_label
positive    764
neutral     433
negative    252
Name: count, dtype: int64

Sentiment score statistics:
count    1449.0000
mean        0.2086
std         0.4369
min        -0.9371
25%         0.0000
50%         0.2382
75%         0.5859
max         0.9848
Name: sentiment_score, dtype: float64


In [4]:
# Aggregate sentiment by bank
print("=== SENTIMENT BY BANK ===\n")

for bank in df['bank'].unique():
    bank_df = df[df['bank'] == bank]
    print(f"{bank}:")
    print(f"  Total reviews: {len(bank_df)}")
    print(f"  Avg sentiment score: {bank_df['sentiment_score'].mean():.4f}")
    print(f"  Positive: {(bank_df['sentiment_label']=='positive').sum()} ({(bank_df['sentiment_label']=='positive').mean()*100:.1f}%)")
    print(f"  Neutral:  {(bank_df['sentiment_label']=='neutral').sum()} ({(bank_df['sentiment_label']=='neutral').mean()*100:.1f}%)")
    print(f"  Negative: {(bank_df['sentiment_label']=='negative').sum()} ({(bank_df['sentiment_label']=='negative').mean()*100:.1f}%)")
    print()

=== SENTIMENT BY BANK ===

Commercial Bank of Ethiopia:
  Total reviews: 456
  Avg sentiment score: 0.2383
  Positive: 258 (56.6%)
  Neutral:  137 (30.0%)
  Negative: 61 (13.4%)

Bank of Abyssinia:
  Total reviews: 498
  Avg sentiment score: 0.1250
  Positive: 219 (44.0%)
  Neutral:  171 (34.3%)
  Negative: 108 (21.7%)

Dashen Bank:
  Total reviews: 495
  Avg sentiment score: 0.2652
  Positive: 287 (58.0%)
  Neutral:  125 (25.3%)
  Negative: 83 (16.8%)



In [5]:
# Mean sentiment score per star rating per bank
print("=== MEAN SENTIMENT BY STAR RATING ===\n")
rating_sentiment = df.groupby(['bank', 'rating'])['sentiment_score'].mean().round(4)
print(rating_sentiment)

=== MEAN SENTIMENT BY STAR RATING ===

bank                         rating
Bank of Abyssinia            1        -0.1796
                             2         0.0222
                             3         0.1954
                             4         0.3241
                             5         0.3223
Commercial Bank of Ethiopia  1        -0.0970
                             2         0.1762
                             3        -0.0750
                             4         0.1693
                             5         0.3908
Dashen Bank                  1        -0.1837
                             2        -0.0104
                             3         0.2543
                             4         0.3382
                             5         0.4456
Name: sentiment_score, dtype: float64


In [1]:
from sklearn.feature_extraction.text import TfidfVectorizer
import re

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['clean_review'] = df['review'].apply(clean_text)

# Define themes and their keywords
THEMES = {
    'Transaction Performance': ['transfer', 'slow', 'fast', 'transaction', 
                                 'payment', 'speed', 'loading', 'delay'],
    'Account Access Issues': ['login', 'password', 'otp', 'access', 
                               'error', 'fingerprint', 'blocked', 'unlock'],
    'App Stability': ['crash', 'bug', 'update', 'fix', 'freeze', 
                       'stop', 'working', 'broken', 'restart'],
    'UI & Design': ['interface', 'design', 'easy', 'simple', 'navigation',
                     'user', 'friendly', 'ui', 'look', 'experience'],
    'Customer Support': ['support', 'service', 'help', 'response', 
                          'agent', 'call', 'staff', 'complaint']
}

def assign_theme(text):
    text = str(text).lower()
    scores = {}
    for theme, keywords in THEMES.items():
        scores[theme] = sum(1 for kw in keywords if kw in text)
    best = max(scores, key=scores.get)
    return best if scores[best] > 0 else 'General Feedback'

df['identified_theme'] = df['clean_review'].apply(assign_theme)

print("✅ Thematic analysis complete!")
print(f"\nTheme distribution:")
print(df['identified_theme'].value_counts())

ModuleNotFoundError: No module named 'sklearn'